# Step B — Marriage Subsidy Calculation
Replicates Section II-B and footnote 13 of Friedberg & Isaac (2024).

**Formula:**  MarriageSubsidy = (T_single_r + T_single_sp) − T_joint

Computed twice:
1. Using **reported** earnings → `msub_total_obs` (used in OLS and descriptives)
2. Using **Lasso-predicted** earnings from Step A → `msub_total_hat` (the IV instrument)

Tax liability = federal income tax + state income tax − EITC − Child Tax Credit

**Requires** `federal_params.py`, `state_params.py`, and `marriage_subsidy.py`
in the same folder as this notebook.

In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd

# Add the folder containing the helper modules to path
HERE = Path.cwd()
sys.path.insert(0, str(HERE))

# ── Update these paths ───────────────────────────────────────────
BASE_DIR = Path.cwd()
IN_PKL   = BASE_DIR / "Replication Project" / "Data" / "Final" / "acs_ssc_predicted_v2.pkl"
OUT_PKL  = BASE_DIR / "Replication Project" / "Data" / "Final" / "acs_ssc_msub_v2.pkl"

from federal_params import (BRACKETS, STANDARD_DEDUCTION, PERSONAL_EXEMPTION,
                             PEP_THRESHOLD, CHILD_TAX_CREDIT,
                             CTC_PHASEOUT_RATE, CTC_PHASEOUT_SINGLE, CTC_PHASEOUT_JOINT,
                             EITC)
from state_params import STATE_TAX

df = pd.read_pickle(IN_PKL)
print(f"Loaded: {df.shape}")

In [ ]:
# ── Federal tax ──────────────────────────────────────────────────
def _apply_brackets(taxable, brackets):
    tax, prev = 0.0, 0.0
    for top, rate in brackets:
        if taxable <= prev: break
        tax += (min(taxable, top) - prev) * rate
        prev = top
    return tax

def _pep(gross, n_exemptions, filing, year):
    """Personal exemption with phase-out."""
    threshold = PEP_THRESHOLD[year][filing]
    exemption = PERSONAL_EXEMPTION[year] * n_exemptions
    if gross <= threshold:
        return exemption
    steps = int(np.ceil((gross - threshold) / 2500))
    return max(exemption - 0.02 * steps * exemption, 0.0)

def _eitc(earned, n_children, filing, year):
    n = min(n_children, 3)
    cr_rate, max_earned, max_credit, po_start_s, po_start_j, po_rate, po_end_s, po_end_j = EITC[year][n]
    credit  = min(earned * cr_rate, max_credit)
    po_start = po_start_j if filing == "joint" else po_start_s
    po_end   = po_end_j   if filing == "joint" else po_end_s
    if earned > po_start:
        credit = max(credit - (earned - po_start) * po_rate, 0.0)
    if earned > po_end:
        credit = 0.0
    return credit

def _ctc(taxable_pre_credits, n_children, filing):
    if n_children == 0: return 0.0
    raw = n_children * CHILD_TAX_CREDIT
    threshold = CTC_PHASEOUT_JOINT if filing == "joint" else CTC_PHASEOUT_SINGLE
    if taxable_pre_credits > threshold:
        raw = max(raw - int(np.ceil((taxable_pre_credits - threshold) / 1000)) * 20, 0.0)
    return raw

def federal_tax(earned, n_children, filing, year):
    earned  = max(earned, 0.0)
    std_ded = STANDARD_DEDUCTION[year][filing]
    n_ex    = 1 if filing == "single" else 2
    pex     = _pep(earned, n_ex, filing, year)
    taxable = max(earned - std_ded - pex, 0.0)
    gross_tax = _apply_brackets(taxable, BRACKETS[year][filing])
    return max(gross_tax - _eitc(earned, n_children, filing, year)
                         - _ctc(earned - std_ded, n_children, filing), 0.0)

print("Federal tax function defined.")

In [ ]:
# ── State tax ────────────────────────────────────────────────────
def state_tax(earned, n_children, filing, statefip, year):
    earned = max(earned, 0.0)
    params = STATE_TAX.get(year, {}).get(int(statefip), {"type": "none"})
    kind   = params.get("type", "none")
    if kind == "none": return 0.0

    std = params["std_ded_joint"]          if filing == "joint" else params["std_ded_single"]
    ex  = params["personal_exempt_joint"]  if filing == "joint" else params["personal_exempt_single"]
    taxable = max(earned - std - ex, 0.0)

    if kind == "flat":
        return taxable * params["rate"]
    bkts = params["brackets_joint"] if filing == "joint" else params["brackets_single"]
    return _apply_brackets(taxable, bkts)

print("State tax function defined.")

In [ ]:
# ── Marriage subsidy for one couple ─────────────────────────────
def couple_subsidy(earn_r, earn_sp, n_children, statefip, year, staterecog):
    """
    Returns dict with msub_fed, msub_state, msub_total,
    and the underlying tax components.
    """
    earn_r, earn_sp = max(earn_r, 0.0), max(earn_sp, 0.0)

    # Assign children to higher earner when filing single
    ch_r, ch_sp = (n_children, 0) if earn_r >= earn_sp else (0, n_children)

    # Single
    fed_r  = federal_tax(earn_r,  ch_r,  "single", year)
    fed_sp = federal_tax(earn_sp, ch_sp, "single", year)
    st_r   = state_tax(earn_r,  ch_r,  "single", statefip, year)
    st_sp  = state_tax(earn_sp, ch_sp, "single", statefip, year)

    # Joint
    joint  = earn_r + earn_sp
    fed_j  = federal_tax(joint, n_children, "joint", year)
    st_j   = (state_tax(joint, n_children, "joint", statefip, year)
               if staterecog == 1 else st_r + st_sp)

    msub_fed   = (fed_r + fed_sp) - fed_j
    msub_state = (st_r  + st_sp)  - (st_j if staterecog else 0.0)
    msub_total = (fed_r + fed_sp + st_r + st_sp) - (fed_j + st_j)

    return dict(t_single_r=fed_r+st_r, t_single_sp=fed_sp+st_sp,
                t_joint=fed_j+st_j,
                msub_fed=msub_fed, msub_state=msub_state,
                msub_total=msub_total)

print("couple_subsidy function defined.")

In [ ]:
# ── Vectorised computation ───────────────────────────────────────
def compute_subsidies(df, earn_r_col, earn_sp_col, suffix):
    results = []
    for row in df.itertuples(index=False):
        res = couple_subsidy(
            earn_r     = float(getattr(row, earn_r_col,  0) or 0),
            earn_sp    = float(getattr(row, earn_sp_col, 0) or 0),
            n_children = int(row.c_children),
            statefip   = int(row.statefip),
            year       = int(row.year),
            staterecog = int(row.staterecog_policy),
        )
        results.append(res)

    res_df = pd.DataFrame(results, index=df.index)

    # Zero out pre-Windsor (year <= 2012): federal not yet recognised
    pre_w = df["preW"] == 1
    res_df.loc[pre_w, "msub_fed"]   = 0.0
    res_df.loc[pre_w, "msub_total"] = res_df.loc[pre_w, "msub_state"]

    out = df.copy()
    for col in ["t_single_r", "t_single_sp", "t_joint",
                "msub_fed", "msub_state", "msub_total"]:
        out[f"{col}_{suffix}"] = res_df[col].values
    return out

print("compute_subsidies defined.")

In [ ]:
# ── Run observed ─────────────────────────────────────────────────
print("Computing observed subsidy (reported earnings)...")
df = compute_subsidies(df, "r_incearn", "sp_incearn", "obs")
print("Done.")

In [ ]:
# ── Run predicted (instrument) ───────────────────────────────────
print("Computing predicted subsidy (Lasso earnings)...")
df = compute_subsidies(df, "hat_incearn_r", "hat_incearn_sp", "hat")
print("Done.")

In [ ]:
# ── Diagnostics vs Table 2 ───────────────────────────────────────
print("="*60)
print("DIAGNOSTICS vs. Table 2")
print("="*60)
for label, mask in [("Married", df["sscouple_mar"]),
                    ("Cohabiting", df["sscouple_coh"])]:
    sub = df[mask]
    print(f"\n  {label} (N={mask.sum():,}):")
    print(f"    Fed+state observed:  mean={sub['msub_total_obs'].mean():8.1f}  "
          f"std={sub['msub_total_obs'].std():8.1f}  "
          f"(paper: mar=442/5117  coh=264/3247)")
    print(f"    Fed+state predicted: mean={sub['msub_total_hat'].mean():8.1f}  "
          f"std={sub['msub_total_hat'].std():8.1f}  "
          f"(paper: mar=68/2219   coh=257/1623)")
    print(f"    Federal observed:    mean={sub['msub_fed_obs'].mean():8.1f}  "
          f"(paper: mar=395  coh=232)")
    print(f"    Federal predicted:   mean={sub['msub_fed_hat'].mean():8.1f}  "
          f"(paper: mar=122  coh=267)")
    print(f"    State observed:      mean={sub['msub_state_obs'].mean():8.1f}  "
          f"(paper: mar=47   coh=32)")

In [ ]:
# ── Save ─────────────────────────────────────────────────────────
df.to_pickle(OUT_PKL)
print(f"Saved to {OUT_PKL}")
new = [c for c in df.columns if c.endswith("_obs") or c.endswith("_hat")]
print(f"New columns ({len(new)}): {new}")